### Havaintoja aineistosta
- Kaikki autot listalla ovat myymättömiä
- Ajallisesti aineisto on erittäin vinoutunut -se kuvaa lähinnä kesä-alkusyksy 2020 markkinatilannetta 
    - Autoista 79 % on listattu 01.06.2020 - 15.09.2020
    - Autoista 92.7 % on listattu 01.01.2020 - 15.09.2020
    - Autoista 98.1% on listattu 1.8.2019 - 15.9.2020
     
- Aikaisemmin listalle tuodut autot ovat valikoitu joukko, ne joita kukaan ei ole ostanut pitkään aikaan. Toki tuon rajan vetämisestä voi diskuteerata, mutta tuolla rajauksella saadaan jo kattava otos koko listasta.

### Edelliset huomioiden analyysin rajaus
- Hinnoittelumalli ja hinnoittelun ennustaminen valitussa ajassa, ei koko ajalta
    - Mitkä tekijät vaikuttavat hinnoitteluun - feature engineering ja importance valideja keissejä edelleen
    - Eli löydetään mitkä tekijät ennustavat myyntihintaa ja opetetaan malli
- Ei sovellu aikasarja- tai trendianalyysiin
    - Kun ei tiedetä myyntiaikaa - polttoaineen hintatrendi ei ole tekijä. Valtaosa listatuista autoista on hinnoiteltu samassa markkinatilanteessa
    - Kyse on hinnoittelusta, ei myyntimääristä - polttoaineen hinta vaikuttaa enemmän kysyntään kuin yksittäisen auton pyyntihintaan

### Eli
- Voidaan lähteä ennustamaan ainoastaan myyntihintaa
- Daysonmarket on täysin laskennallinen arvo, ja vieläpä hyvin yksinkertainen sellainen
- Kannattanee ehkä vielä palata Phase3:een ja kattoa mitä sieltä kannattaa karsia
- Phase4, daysonmarket unohduksiin



In [1]:
import os
from dotenv import load_dotenv
from pathlib import Path
import duckdb
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
#from sklearn.model_selection import GroupKFold
from sklearn.model_selection import train_test_split
import numpy as np
import requests

In [2]:
load_dotenv()

# Luetaan datakansion polku .env-tiedostosta
DATA_PATH = Path(os.getenv("DATA_PATH"))

In [3]:
# =============================================================================
# Polut
# =============================================================================

raw_path         = DATA_PATH / "used_cars_data.parquet"
sample_path_csv  = DATA_PATH / "used_cars_data_sample.csv"
cleaned_path     = DATA_PATH / "used_cars_data_cleaned.parquet"
sample_100k_path = DATA_PATH / "used_cars_data_sample_100k.parquet"

# =============================================================================
# 1. Raakadata: CSV → Parquet (vain kerran)
# =============================================================================

if not raw_path.exists():
    print("Luodaan parquet CSV:stä...")

    duckdb.execute(f"""
        COPY (
            SELECT *
            FROM read_csv_auto(
                '{str(DATA_PATH / "used_cars_data.csv")}',
                types={{'dealer_zip': 'VARCHAR'}}
            )
            ORDER BY random()
            LIMIT 100000
        )
        TO '{str(raw_path)}'
        (FORMAT PARQUET)
    """)

    print(f"Valmis! Tallennettu: {raw_path}")
else:
    print("Raakadata parquet-muodossa löytyy valmiina.")

# =============================================================================
# 2. Sample CSV (vain nopeaa tarkistusta / debugia varten)
#    EI vaikuta varsinaiseen data pipelineen
# =============================================================================


if not sample_100k_path.exists():
    print("Luodaan 100k random sample raw parquetista...")

    duckdb.execute(f"""
        COPY (
            SELECT *
            FROM read_parquet('{str(raw_path)}')
            USING SAMPLE 100000 ROWS
        )
        TO '{str(sample_100k_path)}'
        (FORMAT PARQUET)
    """)

    print(f"Sample luotu: {sample_100k_path}")


# =============================================================================
# 3. Esikäsittely (LAZY, koko data, ei samplea)
# =============================================================================

if cleaned_path.exists():
    # Luetaan jo esikäsitelty data
    df = pl.read_parquet(cleaned_path)

    print(
        f"Ladattu cleaned parquet: "
        f"{df.shape[0]:,} riviä, {df.shape[1]} saraketta"
    )

else:
    print("Ajetaan esikäsittely koko datasetille (lazy)...")

    # Lazy-scan koko parquetille (EI ladota muistiin)
    df_lazy = (
        pl.scan_parquet(raw_path)

        # Valitaan vain tarvittavat sarakkeet
        .select([
            "body_type",
            "daysonmarket",
            "engine_displacement",
            "engine_type",
            "mileage",
            "fuel_type",
            "horsepower",
            "latitude",
            "longitude",
            "major_options",
            "make_name",
            "model_name",
            "price",
            "transmission_display",
            "wheel_system",
            "year",
            "listed_date",
            "length",
        ])

        # Suodatukset
        .filter(pl.col("price").is_between(1_000, 150_000))
        .filter(
            (pl.col("mileage") < 500_000) |
            pl.col("mileage").is_null()
        )
        .filter(pl.col("year").is_between(1990, 2023))
        .filter(pl.col("latitude").is_not_null())
        .filter(pl.col("longitude").is_not_null())

        # Duplikaattien poisto
        .unique()
    )

    # Materialisoidaan vasta lopuksi
    df = df_lazy.collect(streaming=True)

    # Tallennetaan välitulos
    df.write_parquet(cleaned_path)

    print(
        f"Tallennettu cleaned parquet: "
        f"{df.shape[0]:,} riviä, {df.shape[1]} saraketta"
    )

# =============================================================================
# 4. (Valinnainen) Nopeita tarkistuksia
# =============================================================================

print("Pipeline valmis.")

Raakadata parquet-muodossa löytyy valmiina.
Luodaan 100k random sample raw parquetista...
Sample luotu: <PROJECT_DIR>\cars\used_cars_data_sample_100k.parquet
Ladattu cleaned parquet: 2,851,658 riviä, 18 saraketta
Pipeline valmis.


In [7]:
import pandas as pd

df = df.to_pandas()

# 1. Tarkista formaatti
invalid_mask = pd.to_datetime(
    df['listed_date'],
    format="%Y-%m-%d",
    errors='coerce'
).isna()

if invalid_mask.any():
    print("Virheellinen päivämääräformaatti seuraavissa riveissä listed_date sarakkeessa:")
    print(df.loc[invalid_mask, 'listed_date'])
else:
    #print("Kaikki arvot listed_date sarakkeessa ovat muodossa YYYY-MM-DD.")

    # 2. Muunna datetimeksi
    df["listed_date"] = pd.to_datetime(
        df["listed_date"],
        format="%Y-%m-%d"
    )

    # 3. Snapshot-päivä
    snapshot_date = pd.to_datetime("2020-09-10")

    # 4. Laske maksimi päivät
    max_days = (snapshot_date - df["listed_date"]).dt.days

    df["gap_days"] = max_days - df["daysonmarket"]

    # 5. Suodata autot, jotka on todennäköisesti myyty
    sold_before_snapshot = df[df["daysonmarket"] < max_days]

    #print(sold_before_snapshot.shape)

    
unique_count = df["gap_days"].nunique()
#print("gap_days‑sarakkeessa on", unique_count, "eri arvoa")

#print(df["gap_days"].nunique(dropna=False))

#print(df["gap_days"].unique())

df_2016 = df[df["listed_date"].dt.year == 2016]

sorted(df_2016["gap_days"].unique())
    
df_2016["gap_days"].value_counts().sort_index()    

df_2012_2015 = df[
    (df["listed_date"].dt.year >= 2010) &
    (df["listed_date"].dt.year <= 2013)
]
print(df_2012_2015[["listed_date", "daysonmarket", "gap_days"]])


        listed_date  daysonmarket  gap_days
274339   2012-07-15          2979         0
405034   2010-11-29          3573         0
407830   2013-03-21          2730         0
710877   2013-11-05          2500         1
775906   2013-11-07          2499         0
1536092  2013-12-13          2464        -1
1716911  2011-07-23          3337         0
1890748  2013-09-01          2567        -1
1956639  2013-05-01          2688         1
2135963  2013-11-05          2500         1
2171652  2013-12-15          2461         0
2232418  2012-07-19          2976        -1
2429565  2010-11-03          3599         0
2435902  2013-10-11          2527        -1
2679576  2013-11-08          2497         1


In [9]:

count_sep_2020 = df[
    (df["listed_date"] >= "2019-08-01") &
    (df["listed_date"] <= "2020-09-15")
].shape[0]

print(count_sep_2020)


print(f"Prosentuaalinen osuus kaikista: {count_sep_2020/df.shape[0]:.1%}")


2808656
Prosentuaalinen osuus kaikista: 98.5%


In [10]:

start_date = "2018-01-01"
end_date = "2020-09-15"



df_period = df[
    (df["listed_date"] >= start_date) &
    (df["listed_date"] <= end_date)
]

df_period["listed_date"] = pd.to_datetime(df_period["listed_date"])
monthly_counts = (
    df_period["listed_date"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)




total_count = monthly_counts.sum()


monthly_counts.to_csv(
    "monthly_counts2.csv",
    header=["Autojen_lukumäärä"],
    index_label="Kuukausi"
)


print(monthly_counts)
print("\nYhteensä:", total_count)



listed_date
2018-01        233
2018-02        275
2018-03        415
2018-04        380
2018-05        444
2018-06        599
2018-07        742
2018-08        955
2018-09       1052
2018-10       1303
2018-11       1517
2018-12       1901
2019-01       2111
2019-02       2145
2019-03       3554
2019-04       3538
2019-05       4793
2019-06       5735
2019-07       8655
2019-08      15822
2019-09      21651
2019-10      31106
2019-11      40857
2019-12      52552
2020-01      63758
2020-02      86856
2020-03     105565
2020-04      55081
2020-05      77774
2020-06     208145
2020-07     503389
2020-08    1038631
2020-09     507469
Freq: M, Name: count, dtype: int64

Yhteensä: 2849003
